# Human Analyst vs Agent Score Variation (Soccer Dataset)

Reproduces the Silberzahn et al. (2018) many-analysts setting and compares the spread of human analyst responses to agent run-to-run variability on the same task.

**Human analysts**: 29 independent teams each analyzed the soccer dataset and wrote a report. Three raters (Hao, Zach, Chandan) scored each report; the team's final score is the mean of those three.  
**Agent**: 100 runs on soccer/alt (5 perturbations × 20 runs each), all from `scalar_experiments/aggregated_results.csv`.

The **main comparison** is: distribution of 29 team mean scores vs distribution of 100 agent scores.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

BASE_DIR = Path(".").resolve()
HUMAN_CSV = BASE_DIR / "results.csv"
AGENT_CSV = BASE_DIR / ".." / "scalar_experiments" / "aggregated_results" / "aggregated_results.csv"

In [ ]:
# ── Load & clean human scores ──
human_df = pd.read_csv(HUMAN_CSV).dropna(subset=["Team Number"])
human_df["Team Number"] = human_df["Team Number"].astype(int)

HUMAN_RATERS = ["Hao", "Zach", "Chandan"]
human_df = human_df[["Team Number"] + HUMAN_RATERS].dropna()
human_df[HUMAN_RATERS] = human_df[HUMAN_RATERS].astype(float)

# Per-team: mean, std, and range across raters
human_df["mean_score"]  = human_df[HUMAN_RATERS].mean(axis=1)
human_df["std_score"]   = human_df[HUMAN_RATERS].std(axis=1, ddof=1)
human_df["range_score"] = human_df[HUMAN_RATERS].max(axis=1) - human_df[HUMAN_RATERS].min(axis=1)

print(f"Human data: {len(human_df)} teams")
human_df

In [ ]:
# get the number of analyses that have mean human rating above and below 50
num_above_50 = (human_df["mean_score"] > 50).sum()
num_below_50 = (human_df["mean_score"] < 50).sum()
print(f"Proportion of Human Studies with a 'Yes' Conclusion: {num_above_50 / len(human_df):.2f}")
print(f"Proportion of Human Studies with a 'No' Conclusion: {num_below_50 / len(human_df):.2f}")

In [ ]:
# ── Load agent scores (soccer/alt only) ──
agent_df = pd.read_csv(AGENT_CSV, keep_default_na=False)
agent_df = agent_df[(agent_df["dataset"] == "soccer") & (agent_df["distribution"] == "alt")]

# Per perturbation block: std and range across 20 runs
agent_var = (
    agent_df.groupby(["dataset", "distribution", "perturbation"])["response"]
    .agg(n="count", mean="mean", std="std", range=lambda x: x.max() - x.min())
    .reset_index()
)

# Convenience arrays used throughout the notebook
human_team_means = human_df["mean_score"].values
all_agent_scores = agent_df["response"].values

print(f"Agent data: {len(agent_df):,} rows → {len(agent_var)} perturbation blocks")
agent_var

In [ ]:
agent_df

In [ ]:
# get the number of analyses that have mean agentic rating above and below 50
num_above_50 = (agent_df["response"] > 50).sum()
num_below_50 = (agent_df["response"] < 50).sum()
print(f"Proportion of Agentic Studies with a 'Yes' Conclusion: {num_above_50 / len(agent_df):.2f}")
print(f"Proportion of Agentic Studies with a 'No' Conclusion: {num_below_50 / len(agent_df):.2f}")

In [ ]:
# ── Summary statistics: spread of the two distributions ──
summary = pd.DataFrame({
    "source":   ["Human analysts (29 teams)", "Agent (100 runs, soccer/alt)"],
    "n":        [len(human_team_means),        len(all_agent_scores)],
    "mean":     [human_team_means.mean(),      all_agent_scores.mean()],
    "median":   [np.median(human_team_means),  np.median(all_agent_scores)],
    "std":      [human_team_means.std(),        all_agent_scores.std()],
    "min":      [human_team_means.min(),        all_agent_scores.min()],
    "max":      [human_team_means.max(),        all_agent_scores.max()],
})
summary

In [ ]:
# ── Main figure: boxplot of human team means vs all agent scores ──
human_team_means = human_df["mean_score"].values
all_agent_scores = agent_df["response"].values

LABEL_FS = 20
TICK_FS  = 16

box_df = pd.DataFrame({
    "score":  list(human_team_means) + list(all_agent_scores),
    "source": ["Human Analysts"] * len(human_team_means) + ["Agent (GPT-5.2-Codex)"] * len(all_agent_scores),
})

fig, ax = plt.subplots(figsize=(7, 5))
palette = {"Human Analysts": "#6A9E73", "Agent (GPT-5.2-Codex)": "#9B6B9E"}
sns.boxplot(data=box_df, x="source", y="score", hue="source",
            palette=palette, width=0.4, ax=ax)
sns.stripplot(data=box_df, x="source", y="score", hue="source",
              palette=palette, alpha=0.4, size=5, jitter=True, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Likert Score (0–100)", fontsize=LABEL_FS)
ax.set_ylim(-5, 105)
# put tick marks on x-axis for labels
ax.set_xticks([0, 1])
ax.set_xticklabels(["Human Analysts", "Agent"], fontsize=LABEL_FS)
plt.tight_layout()
# save figure to 'images' subdir within current directory
plt.savefig("images/human_vs_agent_response_distributions.png",
            dpi=600,
            bbox_inches="tight")
plt.show()

In [ ]:
# get the median yes and no score for the agentic analyses
agent_yes_median = agent_df[agent_df["response"] > 50]["response"].median()
agent_no_median = agent_df[agent_df["response"] < 50]["response"].median()
print(f"Median 'Yes' Score for Agentic Analyses: {agent_yes_median:.2f}")
print(f"Median 'No' Score for Agentic Analyses: {agent_no_median:.2f}")

In [ ]:
# ── KDE: human analysts vs per-perturbation agent distributions ──
perturbations = agent_df["perturbation"].unique()
pert_palette = sns.color_palette("Reds", n_colors=len(perturbations) + 1)[1:]

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax = axes[0]
sns.kdeplot(human_team_means, ax=ax, label=f"Human analysts ({len(human_team_means)} teams)",
            color="steelblue", fill=True, alpha=0.25, linewidth=2)
for pert, color in zip(sorted(perturbations), pert_palette):
    scores = agent_df.loc[agent_df["perturbation"] == pert, "response"].values
    sns.kdeplot(scores, ax=ax, label=pert, color=color, linewidth=1.5)
ax.set_ylabel("Density")
ax.set_title("Score Distributions: Human Analysts vs Agent by Perturbation")
ax.legend(fontsize=8, loc="upper left")

ax = axes[1]
bins = range(0, 105, 5)
ax.hist(human_team_means, bins=bins, alpha=0.5, color="steelblue",
        label="Human analysts", density=True)
for pert, color in zip(sorted(perturbations), pert_palette):
    scores = agent_df.loc[agent_df["perturbation"] == pert, "response"].values
    ax.hist(scores, bins=bins, alpha=0.4, color=color, label=pert, density=True)
ax.set_xlabel("Score (0–100)")
ax.set_ylabel("Density")
ax.legend(fontsize=8, loc="upper left")

plt.xlim(0, 100)
plt.tight_layout()
plt.show()

---
## Appendix: Inter-Rater Agreement

The three raters (Hao, Zach, Chandan) scored each team's report independently. The plots below show how consistently they agreed, justifying the use of their mean as the team's final score.

In [ ]:
# ── Inter-rater agreement: individual rater vs team mean ──
rater_rename = {"Hao": "Rater 1", "Zach": "Rater 2", "Chandan": "Rater 3"}
rater_palette = {"Rater 1": "#6A9E73", "Rater 2": "#C75E6A", "Rater 3": "#9B6B9E"}

melted = human_df.melt(id_vars=["Team Number", "mean_score"],
                        value_vars=HUMAN_RATERS, var_name="rater", value_name="score")
melted["rater"] = melted["rater"].map(rater_rename)

fig, ax = plt.subplots(figsize=(6, 5))
for rater, grp in melted.groupby("rater"):
    ax.scatter(grp["mean_score"], grp["score"], label=rater,
               color=rater_palette[rater], alpha=0.75, s=55)
ax.plot([-5, 105], [-5, 105], "k--", linewidth=1)
ax.set_xlim(-5, 105); ax.set_ylim(-5, 105)
ax.set_xlabel("Mean Human Score", fontsize=15)
ax.set_ylabel("Individual Rater Score", fontsize=15)
# make axis ticks bigger
ax.tick_params(labelsize=14)
ax.legend(fontsize=14)
plt.tight_layout()
plt.savefig("images/individual_rater_vs_team_mean.png",
            dpi=500,
            bbox_inches="tight")
plt.show()

In [ ]:
# ── Deviation from team mean score, per rater ──
melted["deviation"] = melted["score"] - melted["mean_score"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=melted, x="rater", y="deviation", hue="rater",
            palette=rater_palette, ax=ax, width=0.4, order=list(rater_rename.values()))
sns.stripplot(data=melted, x="rater", y="deviation", hue="rater",
              palette=rater_palette, ax=ax, alpha=0.5, size=5,
              jitter=True, order=list(rater_rename.values()))
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("")
ax.set_ylabel("(Individual Score) − (Mean Score)", fontsize=15)
ax.tick_params(labelsize=14)
plt.tight_layout()
plt.savefig("images/individual_rater_deviation.png",
            dpi=500,
            bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-team bar chart: individual rater scores sorted by team mean ──
sorted_df = human_df.sort_values("mean_score").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(sorted_df))
width = 0.22

# make rater to column name mapping
rater_nums = {
        "Rater 1": "Hao",
        "Rater 2": "Zach",
        "Rater 3": "Chandan",
}

for i, (rater, color) in enumerate(rater_palette.items()):
    ax.bar(x + i * width, sorted_df[rater_nums[rater]], width=width,
           label=rater, color=color, alpha=0.8)

ax.plot(x + width, sorted_df["mean_score"], "ko--",
        markersize=4, linewidth=1, label="Mean Rating", zorder=5)

ax.set_xticks(x + width)
ax.set_xticklabels([f"T{t}" for t in sorted_df["Team Number"]])
ax.set_ylabel("Likert Score (0–100)", fontsize=15)
ax.set_xlabel("Human-Performed Analyses", fontsize=15)
ax.legend(fontsize=14)
ax.set_ylim(0, 110)
ax.tick_params(axis='y', labelsize=14)
ax.tick_params(axis='x', labelsize=10)
plt.tight_layout()
plt.savefig("images/scores_per_team.png",
            dpi=500,
            bbox_inches="tight")
plt.show()

In [ ]:
# ── Statistical comparison: human analyst spread vs agent spread ──
# Levene's test: are the variances different?
levene_stat, p_levene = stats.levene(human_team_means, all_agent_scores)
# Welch t-test: are the means different?
t_stat, p_ttest = stats.ttest_ind(human_team_means, all_agent_scores, equal_var=False)

print("Levene's test for equal variances:")
print(f"  W = {levene_stat:.3f},  p = {p_levene:.4f}")
print()
print("Welch t-test (difference in means):")
print(f"  t = {t_stat:.3f},  p = {p_ttest:.4f}")
print()

# Ratio of stds as a simple effect size
std_ratio = human_team_means.std() / all_agent_scores.std()
print(f"Std ratio (human / agent) = {std_ratio:.2f}")